# GRADE: Rating Certainty of Evidence — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicai/learn-notebooks/blob/main/notebooks/evidence-synthesis/15-2/grade-certainty.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicai/learn-notebooks/blob/main/notebooks/evidence-synthesis/15-2/grade-certainty.ipynb)

> Companion to the **GRADE: Rating Certainty of Evidence** article. GRADE rates how much to
> trust each outcome's estimate on four levels — **HIGH, MODERATE, LOW, VERY LOW**. You pick a
> **starting point** by study design, **rate down** for five problems, **rate up** (observational
> only) for three strengths, then clamp. The judgments are human; **the arithmetic is code** —
> so watch the level fall out deterministically, one print at a time.

## What you'll build

1. A deterministic `grade_certainty()` engine — judgments in, a level out.
2. The **four worked examples** from the article (RCT to LOW, observational up to MODERATE, a blocked upgrade, an RCT clamped at VERY LOW).
3. A per-outcome panel table (mortality HIGH vs quality-of-life LOW) so you *see* why certainty is rated per outcome.

**No API key needed.** This is a pure-Python rule engine (no installs, no network) — the reader watches every judgment become a level.

### Setup (nothing to install)

Pure Python — the standard library `dataclasses` is all we need.

In [1]:
from dataclasses import dataclass, field
print("ready — pure Python, no API key, no installs")

ready — pure Python, no API key, no installs


## The scale and the vocabulary

Certainty is a slider on a **1-4** scale: `1 = VERY LOW ... 4 = HIGH`. There are exactly five
**rate-down** domains and three **rate-up** factors — a fixed vocabulary, so a typo can't
silently vanish.

In [2]:
# 1 = VERY LOW ... 4 = HIGH
_LEVELS = {1: "VERY LOW", 2: "LOW", 3: "MODERATE", 4: "HIGH"}

# the five rate-down domains and the three rate-up factors (fixed vocab)
DOWNGRADE_DOMAINS = ("risk_of_bias", "inconsistency", "indirectness",
                     "imprecision", "publication_bias")
UPGRADE_FACTORS = ("large_effect", "dose_response", "confounding_reduces_effect")

print("levels        :", _LEVELS)
print("rate-DOWN (5) :", DOWNGRADE_DOMAINS)
print("rate-UP   (3) :", UPGRADE_FACTORS)

levels        : {1: 'VERY LOW', 2: 'LOW', 3: 'MODERATE', 4: 'HIGH'}
rate-DOWN (5) : ('risk_of_bias', 'inconsistency', 'indirectness', 'imprecision', 'publication_bias')
rate-UP   (3) : ('large_effect', 'dose_response', 'confounding_reduces_effect')


## The engine — judgments in, a level out

The human (or an LLM) supplies the **judgments**: study design, how many levels each rate-down
domain costs, and which rate-up factors apply. `grade_certainty()` does only the fixed
arithmetic and rule enforcement, so *judgments -> level* is 100% reproducible.

Four rules:

```
RULE 1  start = HIGH(4) if randomized else LOW(2)
RULE 2  each rate-DOWN domain contributes 0, -1 (serious), or -2 (very serious)
RULE 3  rate-UP applies ONLY to observational evidence AND ONLY if total downgrades == 0
RULE 4  score = start - downgrades + upgrades, then CLAMP to [1 (VERY LOW) .. 4 (HIGH)]
```

In [3]:
@dataclass
class GradeInput:
    randomized: bool                                 # True -> start HIGH, False -> start LOW
    downgrades: dict = field(default_factory=dict)   # e.g. {"risk_of_bias": 1, "imprecision": 1}
    upgrades: dict = field(default_factory=dict)     # e.g. {"large_effect": 1}


def grade_certainty(inp: GradeInput) -> dict:
    """Return the GRADE certainty level and an audit trail."""
    # --- validate the vocabulary so typos can't silently vanish ---
    for k, v in inp.downgrades.items():
        if k not in DOWNGRADE_DOMAINS:
            raise ValueError(f"unknown downgrade domain: {k}")
        if v not in (0, 1, 2):
            raise ValueError(f"{k}: downgrade must be 0, 1 (serious) or 2 (very serious)")
    for k, v in inp.upgrades.items():
        if k not in UPGRADE_FACTORS:
            raise ValueError(f"unknown upgrade factor: {k}")
        if v not in (0, 1, 2):
            raise ValueError(f"{k}: upgrade must be 0, 1 or 2")

    # RULE 1: starting point by design
    start = 4 if inp.randomized else 2          # HIGH vs LOW
    total_down = sum(inp.downgrades.values())

    # RULE 3: upgrades only for observational evidence with NO downgrades
    up_allowed = (not inp.randomized) and total_down == 0
    total_up = sum(inp.upgrades.values()) if up_allowed else 0
    up_blocked = bool(inp.upgrades) and not up_allowed

    # RULE 4: combine, then clamp into [1, 4]
    raw = start - total_down + total_up
    score = max(1, min(4, raw))

    return {
        "level": _LEVELS[score],
        "score": score,
        "start": _LEVELS[start],
        "total_downgrades": total_down,
        "total_upgrades": total_up,
        "upgrades_blocked": up_blocked,   # True if caller asked for an illegal upgrade
        "clamped": raw != score,
    }

print("grade_certainty() defined — the human judges, the code counts")

grade_certainty() defined — the human judges, the code counts


## Worked example A — RCTs rated down to LOW

A meta-analysis of randomized trials of a new drug vs placebo for preventing stroke
(outcome: non-fatal stroke). Starts **HIGH**, loses **-1 risk of bias** (open-label,
unclear adjudication) and **-1 imprecision** (only 90 events, CI 0.55-0.98 nearly touches 1.0).

`4 (HIGH) - 1 - 1 = 2 -> LOW`.

In [4]:
a = grade_certainty(GradeInput(
    randomized=True,
    downgrades={"risk_of_bias": 1, "imprecision": 1}))

print("Design       : randomized  -> start", a["start"])
print("Downgrades   :", a["total_downgrades"], "(risk_of_bias -1, imprecision -1)")
print("Upgrades     :", a["total_upgrades"], "(n/a for RCTs)")
print("A:", a["start"], "->", a["level"])       # HIGH -> LOW
print(a)

Design       : randomized  -> start HIGH
Downgrades   : 2 (risk_of_bias -1, imprecision -1)
Upgrades     : 0 (n/a for RCTs)
A: HIGH -> LOW
{'level': 'LOW', 'score': 2, 'start': 'HIGH', 'total_downgrades': 2, 'total_upgrades': 0, 'upgrades_blocked': False, 'clamped': False}


## Worked example B — observational evidence rated UP to MODERATE

Cohort studies of a fall-prevention program (outcome: hip fracture). Starts **LOW**, has
**no serious problems** (total downgrades = 0), and the effect is **large** (RR ~ 0.45) -> **+1**.
The upgrade is allowed *because* it's observational **and** has zero downgrades.

`2 (LOW) + 1 = 3 -> MODERATE`.

In [5]:
b = grade_certainty(GradeInput(
    randomized=False,
    upgrades={"large_effect": 1}))

print("Design       : observational -> start", b["start"])
print("Downgrades   :", b["total_downgrades"], "-> upgrade lane is OPEN")
print("Upgrades     :", b["total_upgrades"], "(large_effect +1)")
print("B:", b["start"], "->", b["level"])       # LOW -> MODERATE

Design       : observational -> start LOW
Downgrades   : 0 -> upgrade lane is OPEN
Upgrades     : 1 (large_effect +1)
B: LOW -> MODERATE


## Worked example C — the upgrade is BLOCKED

Same cohort evidence, same large effect — but now it also has a **serious risk-of-bias**
problem. You never rescue evidence that has serious flaws, so the rate-up is **gated off**
(downgrades != 0). The `large_effect: 1` we passed in is ignored *and* flagged.

`2 (LOW) - 1 + 0 = 1 -> VERY LOW`, `upgrades_blocked = True`.

In [6]:
c = grade_certainty(GradeInput(
    randomized=False,
    downgrades={"risk_of_bias": 1},
    upgrades={"large_effect": 1}))

print("Design       : observational -> start", c["start"])
print("Downgrades   :", c["total_downgrades"], "-> upgrade lane is CLOSED")
print("Upgrades used:", c["total_upgrades"], "(the +1 we asked for was ignored)")
print("C:", c["level"], "| upgrades_blocked =", c["upgrades_blocked"])  # VERY LOW / True

Design       : observational -> start LOW
Downgrades   : 1 -> upgrade lane is CLOSED
Upgrades used: 0 (the +1 we asked for was ignored)
C: VERY LOW | upgrades_blocked = True


## Worked example D — clamp at VERY LOW (never below)

An RCT with three serious problems: `-1 risk of bias, -1 inconsistency, -2 imprecision`.
Raw arithmetic is `4 - 4 = 0`, but the final `clamp` stops the slider at **VERY LOW (1)** —
there is no "below very low."

`clamp(4 - 4, 1, 4) = 1 -> VERY LOW`, `clamped = True`.

In [7]:
d = grade_certainty(GradeInput(
    randomized=True,
    downgrades={"risk_of_bias": 1, "inconsistency": 1, "imprecision": 2}))

print("Raw score would be : 4 -", d["total_downgrades"], "=", 4 - d["total_downgrades"])
print("D:", d["level"], "| clamped =", d["clamped"])   # VERY LOW / True

Raw score would be : 4 - 4 = 0
D: VERY LOW | clamped = True


## Certainty is rated PER OUTCOME

The same drug, the same trials — but two outcomes get **two different certainties**, because the
evidence for each has different flaws. Mortality is objective and clean (**HIGH**); a
self-reported quality-of-life outcome is unblinded and imprecise (**-1 risk of bias, -1
imprecision -> LOW**). A guideline that lumped them together would overstate the soft outcome.

In [8]:
panel = [
    ("All-cause mortality (8 RCTs)",        GradeInput(randomized=True)),
    ("Quality of life, self-report (3 RCTs)", GradeInput(randomized=True,
        downgrades={"risk_of_bias": 1, "imprecision": 1})),
]

print(f"{'Outcome':<40}{'Start':<10}{'Final'}")
print("-" * 62)
for name, inp in panel:
    r = grade_certainty(inp)
    print(f"{name:<40}{r['start']:<10}{r['level']}")

print("\nSame drug, same review -> two certainties, because certainty is PER OUTCOME.")

Outcome                                 Start     Final
--------------------------------------------------------------
All-cause mortality (8 RCTs)            HIGH      HIGH
Quality of life, self-report (3 RCTs)   HIGH      LOW

Same drug, same review -> two certainties, because certainty is PER OUTCOME.


## Recap

- **Certainty = confidence the true effect is close to the estimate** — HIGH / MODERATE / LOW / VERY LOW, rated **per outcome**.
- **Start** by design (RCT HIGH, observational LOW), **rate down** for five domains, **rate up** (observational-only, no-downgrades-gated) for three factors, then **clamp**.
- You watched the guardrails bite: an illegal upgrade was **blocked and flagged**, and an over-downgraded RCT **clamped** at VERY LOW.
- Judgments come in as data; the level comes out by rule — that is why the arithmetic belongs in deterministic code.

← Back to the **GRADE: Rating Certainty of Evidence** article for the full explanation, the domain tables, and interview Q&A.